In [ ]:
from src.embeddings import get_bge_small_embed
from src.ingestion import load_documents,chunking
from src.retreival import make_query_engine
from src.Chromadb_init import build_chroma_index

In [ ]:
from src.ingestion import pdfs
import time
docs=load_documents(pdfs,80)
embed=get_bge_small_embed()
ablation_configs=[
    ("384_50",384,50,"legal_bge_384"),
    ("512_100",512,100,"legal_bge_512"),
    ("768_150",768,150,"legal_bge_150")
]
indexes={}
for name,chunk_size,chunk_overlap,collection_name in ablation_configs:
    nodes=chunking(docs,chunk_size,chunk_overlap)
    idx=build_chroma_index(nodes,f"../indexes/chroma_bge_small{name}",collection_name,embed)
    t0 = time.time()
    indexes[name]=idx
    print(f"{name}: {len(nodes)} chunks | {(time.time()-t0)/60:.1f} min")

In [ ]:
from src.reranker import get_cross_encoder_reranker

questions=[
    "what category cases mostly occurred in 1975?",
    "How many civil and criminal cases are solved? ",
    "Who is the most successful judge gave decision in minimal time?",
    "How much money laundering have happened and money value too?"
]
reranker=get_cross_encoder_reranker()

comparison=[]

for q in questions:
    row={"question":q}
    for name,idx in indexes.items():
        qe=idx.as_query_engine(
            similarity_top_k=20,
            node_postprocessors=[reranker]

        )
        t0=time.time()
        resp=qe.query(q)
        row[f"answer_{name}"]=str(resp)[:400]
        row[f"latency_{name}"]=round(time.time()-t0,2)
        comparison.append(row)
import pandas as pd
df=pd.DataFrame(comparison)
df.to_csv("../results/week2_ablation.csv",index=False)
df[["question","answer_384_50","answer_512_100","answer_768_150"]].head(3)



In [ ]:
best_idx=indexes["512_100"]

#without reranker
qe_base=best_idx.as_query_engine(similarity_top_k=5)

#with reranker
qe_reranker=best_idx.as_query_engine(similarity_top_k=20,node_postprocessors=[reranker])

before_after=[]

for q in questions:
    base_resp=qe_base.query(q)
    reranked_resp=qe_reranker.query(q)
    before_after.append(
        {
            "question":q,
            "baseline_ans":str(base_resp)[:400],
            "reranked_ans":str(reranked_resp)[:400]
        }
    )

pd.DataFrame(before_after).to_csv("../results/week2_before_after.csv",index=False)
